# Day 4 — LLM Orchestration with LangChain & LangGraph
## Industrial AI & LLM Training Program

**Session:** Day 4 of 5  
**Time:** 20:00–22:30 WIB  
**Prerequisites:** Day 1 (text mining), Day 2 (LLM API), Day 3 (RAG pipeline)  

### What we build tonight

| Section | Title | Key output |
|---------|-------|------------|
| 0 | Setup | Libraries, provider detection |
| 1 | Chains in LangChain (LCEL) | `ticket_chain` — prompt → model → parser |
| 2 | Tools & Tool Use | Custom tools: ticket_lookup, equipment_status, sop_search |
| 3 | ReAct Agents | `create_react_agent` with tool list |
| 4 | Memory & Stateful Interactions | Multi-turn support agent conversation |
| 5 | Introduction to LangGraph | Simple linear StateGraph |
| 6 | Multi-Step & Conditional Workflow | Triage graph with conditional routing |
| 7 | Error Handling & Human-in-the-Loop | Retry logic, interrupt_before, approval node |
| 8 | Use Case: Autonomous Support Agent | Full pipeline: classify → RAG → draft → escalate |
| 9 | Next Steps | Day 5 preview, homework |

**Building on Day 3's RAG pipeline** — the `rag_query()` function from Day 3  
is integrated as a tool in the Day 4 agent. Day 5 covers deployment with vLLM and Ollama.

In [ ]:
# CELL 0-A: Install required libraries (run this if you get ImportError below)
# Uncomment and run if needed:

# !pip install langchain langchain-anthropic langchain-openai langgraph python-dotenv
# !pip install anthropic openai requests numpy pandas
# !pip install sentence-transformers faiss-cpu  # for Day 3 RAG integration

print('Cell 0-A: Install cell — uncomment lines above if needed')
print('Core requirement: langchain + langgraph')
print('LLM backend: langchain-anthropic (or langchain-openai)')

In [ ]:
# CELL 0-B: Imports and provider detection
import os, json, time, warnings, re
import numpy as np
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# --- LangChain core (always available) ---
LANGCHAIN_AVAILABLE = False
try:
    from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
    from langchain_core.tools import tool
    LANGCHAIN_AVAILABLE = True
    print('✓ langchain-core: available')
except ImportError:
    print('⚠  langchain-core not installed → using mock implementations')

# --- LangGraph ---
LANGGRAPH_AVAILABLE = False
try:
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver
    LANGGRAPH_AVAILABLE = True
    print('✓ langgraph: available')
except ImportError:
    print('⚠  langgraph not installed → using mock graph implementations')

# --- LLM provider detection (same pattern as Day 2 & Day 3) ---
PROVIDER = 'mock'

anthropic_key = os.getenv('ANTHROPIC_API_KEY', '')
openai_key = os.getenv('OPENAI_API_KEY', '')

if anthropic_key and anthropic_key != 'your-key-here':
    try:
        from langchain_anthropic import ChatAnthropic
        llm = ChatAnthropic(
            model='claude-haiku-4-5-20251001',
            api_key=anthropic_key,
            temperature=0.0,
            max_tokens=512
        )
        PROVIDER = 'anthropic'
        print(f'✓ LLM provider: Anthropic (claude-haiku-4-5-20251001)')
    except ImportError:
        print('⚠  langchain-anthropic not installed, trying openai...')

if PROVIDER == 'mock' and openai_key and openai_key != 'your-key-here':
    try:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model='gpt-4o-mini', api_key=openai_key, temperature=0.0)
        PROVIDER = 'openai'
        print(f'✓ LLM provider: OpenAI (gpt-4o-mini)')
    except ImportError:
        print('⚠  langchain-openai not installed')

if PROVIDER == 'mock':
    print('ℹ  LLM provider: Mock (no API key or langchain packages detected)')
    print('   All cells will run with deterministic mock responses.')

print(f'\n--- Environment summary ---')
print(f'LangChain available : {LANGCHAIN_AVAILABLE}')
print(f'LangGraph available : {LANGGRAPH_AVAILABLE}')
print(f'LLM provider        : {PROVIDER}')

In [ ]:
# CELL 0-C: Mock LLM and mock LangChain components for offline use
# If LANGCHAIN_AVAILABLE is False, these stubs let every downstream cell run.

class MockMessage:
    def __init__(self, content): self.content = content

class MockLLM:
    """Deterministic mock LLM — returns canned responses based on prompt keywords."""
    def invoke(self, messages):
        if isinstance(messages, list):
            text = ' '.join(str(m) for m in messages).lower()
        else:
            text = str(messages).lower()
        # Routing responses
        if 'kategori' in text or 'categorize' in text or 'classify' in text:
            return MockMessage('Kategori: Mechanical | Prioritas: Tinggi | Tindakan: Periksa bearing dan seal pompa P-101. Jadwalkan WO preventive maintenance.')
        if 'loto' in text or 'lockout' in text:
            return MockMessage('Prosedur LOTO P-101: (1) Notifikasi control room. (2) Tutup valve isolasi. (3) Pasang lock dan tag. (4) Verifikasi zero energy. [Source: SOP-MECH-001]')
        if 'status' in text or 'equipment' in text:
            return MockMessage('Status P-101: Operational — suhu 72°C (normal), tekanan 4.2 bar, vibrasi 3.1 mm/s. Maintenance terjadwal: 15 Mar 2025.')
        if 'escalat' in text or 'eskalasi' in text:
            return MockMessage('ESKALASI DIPERLUKAN: Tiket ini membutuhkan persetujuan supervisor sebelum tindakan. Hubungi: supervisor@plant.id')
        if 'draft' in text or 'response' in text or 'balasan' in text:
            return MockMessage('Yth. Tim Teknik,\n\nTerima kasih atas laporan Anda. Berdasarkan analisis kami:\n- Kategori: Mechanical/Safety\n- Tindakan: WO #WO-2025-0342 telah dibuat\n- SOP rujukan: SOP-MECH-001 (LOTO Procedure)\n\nTim akan menindaklanjuti dalam 2 jam. [Source: SOP-MECH-001]')
        return MockMessage('Permintaan diterima. Sedang memproses berdasarkan data yang tersedia dari knowledge base industrial.')
    
    def bind_tools(self, tools): return self  # mock tool binding

if PROVIDER == 'mock':
    llm = MockLLM()

# Mock LangChain components if library not available
if not LANGCHAIN_AVAILABLE:
    class StrOutputParser:
        def __init__(self): pass
        def invoke(self, msg):
            return msg.content if hasattr(msg, 'content') else str(msg)
    
    class ChatPromptTemplate:
        @classmethod
        def from_messages(cls, messages):
            obj = cls()
            obj._messages = messages
            return obj
        def invoke(self, variables):
            return [str(m) for m in self._messages]

    def tool(func): return func  # decorator passthrough

print('✓ Mock components ready')
print('  All sections will run regardless of package availability.')

---
## Section 1 — Chains in LangChain (LCEL)

### What is LCEL?

**LangChain Expression Language (LCEL)** uses the pipe operator `|` to chain components:

```
prompt | model | parser
```

Every component implements the **Runnable** protocol: `.invoke()`, `.stream()`, `.batch()`.  
You can chain any Runnable with `|` — the output of left becomes input of right.

**Day 3 comparison:** In Day 3, we manually formatted prompts and called `chat()`. Today,  
LangChain components handle formatting, calling the LLM, and parsing output automatically.

| Day 3 (manual) | Day 4 (LCEL) |
|---|---|
| `prompt_str = f"Classify: {ticket}"` | `prompt = ChatPromptTemplate.from_messages([...])` |
| `response = chat([{'role':'user','content':prompt_str}])` | `chain = prompt \| llm \| StrOutputParser()` |
| `result = response` | `result = chain.invoke({'ticket': ticket})` |

In [ ]:
# CELL 1-A: Build a simple LCEL chain for ticket classification

# --- Prompt template ---
CLASSIFY_SYSTEM = """Kamu adalah AI support agent untuk fasilitas manufaktur industri.
Klasifikasikan tiket support berikut dan berikan tindakan yang disarankan.
Format output:
Kategori: [Mechanical/Electrical/Safety/SAP/IT]
Prioritas: [Kritis/Tinggi/Sedang/Rendah]
Tindakan: [Langkah spesifik yang disarankan]"""

if LANGCHAIN_AVAILABLE:
    classify_prompt = ChatPromptTemplate.from_messages([
        ('system', CLASSIFY_SYSTEM),
        ('human', 'Tiket {ticket_id}: {ticket_text}')
    ])
    
    parser = StrOutputParser()
    
    # Build chain with pipe operator
    ticket_chain = classify_prompt | llm | parser
    
    print('Chain built: classify_prompt | llm | parser')
    print(f'Chain type: {type(ticket_chain).__name__}')
else:
    # Mock chain for offline use
    class MockChain:
        def invoke(self, variables):
            return llm.invoke([variables.get('ticket_text', '')]).content
        def batch(self, inputs):
            return [self.invoke(inp) for inp in inputs]
    ticket_chain = MockChain()
    print('Chain built: MockChain (LCEL not available)')

# --- Test with a single maintenance ticket ---
test_ticket = {
    'ticket_id': 'M01',
    'ticket_text': 'Pompa sentrifugal P-101 mengeluarkan bunyi berisik dan getaran tinggi. Bearing mungkin aus.'
}

print(f'\nTesting chain on ticket M01...')
result = ticket_chain.invoke(test_ticket)
print(f'\nOutput:\n{result}')

In [ ]:
# CELL 1-B: Batch processing — run chain on multiple tickets

sample_tickets = [
    {'ticket_id': 'M01', 'ticket_text': 'Pompa sentrifugal P-101 mengeluarkan bunyi berisik dan getaran tinggi. Bearing mungkin aus.'},
    {'ticket_id': 'K05', 'ticket_text': 'Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan.'},
    {'ticket_id': 'E03', 'ticket_text': 'Motor penggerak conveyor C-301 tidak mau start. Indikator trip di panel MCC.'},
    {'ticket_id': 'S01', 'ticket_text': 'Error ME21N: vendor master belum disetujui procurement. PO tidak bisa dibuat.'},
    {'ticket_id': 'N04', 'ticket_text': 'Koneksi VPN ke plant network terputus setiap 2 jam. Teknisi remote tidak bisa akses DCS.'},
]

print('Batch processing 5 tickets...\n')
start = time.time()
results = ticket_chain.batch(sample_tickets)
elapsed = time.time() - start

for ticket, result in zip(sample_tickets, results):
    print(f"--- Ticket {ticket['ticket_id']} ---")
    print(result[:200])
    print()

print(f'Batch complete: {len(results)} tickets in {elapsed:.2f}s ({elapsed/len(results):.2f}s per ticket)')

In [ ]:
# CELL 1-C: Chain inspection — understand what LCEL builds

print('=== Chain Inspection ===')
print(f'Chain type       : {type(ticket_chain).__name__}')

if LANGCHAIN_AVAILABLE and hasattr(ticket_chain, 'steps'):
    for i, step in enumerate(ticket_chain.steps):
        print(f'Step {i}           : {type(step).__name__}')
elif LANGCHAIN_AVAILABLE:
    print('Steps            : classify_prompt | llm | StrOutputParser')
else:
    print('Steps            : MockChain (LCEL not available)')

print()
print('=== Runnable Protocol ===')
print('Every LangChain component has:')
print('  .invoke(input)   → single call')
print('  .batch([inputs]) → parallel calls')
print('  .stream(input)   → streaming tokens')
print()
print('=== LCEL Advantages ===')
print('1. Composability  : chain any Runnable with |')
print('2. Streaming      : .stream() works on entire chain')
print('3. Async          : .ainvoke() / .astream() built-in')
print('4. Observability  : LangSmith tracing with zero extra code')
print('5. Fallbacks      : chain.with_fallbacks([backup_llm])')

---
## Section 2 — Tools & Tool Use

### What are LangChain Tools?

A **tool** is a function the LLM can call. You define it with `@tool`; LangChain generates  
the JSON schema automatically. The LLM sees the schema, decides to call the tool, and  
LangChain executes the function and returns the result.

```
@tool
def ticket_lookup(ticket_id: str) -> str:
    """Look up a maintenance ticket by ID. Returns ticket details."""
    ...
```

**Industrial use cases for tools:**
- `ticket_lookup(id)` — fetch ticket from CMMS
- `equipment_status(equipment_id)` — query equipment health from historian
- `sop_search(query)` — semantic search in knowledge base (Day 3 RAG!)
- `create_work_order(equipment, description)` — write to SAP PM
- `send_escalation(ticket_id, reason)` — notify supervisor

In [ ]:
# CELL 2-A: Define industrial support tools

# --- Synthetic equipment and ticket database ---
EQUIPMENT_DB = {
    'P-101': {'name': 'Centrifugal Pump P-101', 'status': 'operational', 'temp_C': 72, 'pressure_bar': 4.2, 'vibration_mms': 3.1, 'next_pm': '2025-03-15'},
    'K-202': {'name': 'Compressor K-202', 'status': 'warning', 'temp_C': 95, 'pressure_bar': 8.7, 'vibration_mms': 6.8, 'next_pm': '2025-02-28'},
    'C-301': {'name': 'Conveyor C-301', 'status': 'fault', 'temp_C': 45, 'pressure_bar': 0, 'vibration_mms': 0, 'next_pm': '2025-03-01'},
    'HE-401': {'name': 'Heat Exchanger HE-401', 'status': 'operational', 'temp_C': 120, 'pressure_bar': 6.1, 'vibration_mms': 1.2, 'next_pm': '2025-04-10'},
}

TICKET_DB = {
    'M01': {'text': 'Pompa sentrifugal P-101 bunyi berisik, getaran tinggi.', 'category': 'Mechanical', 'priority': 'Tinggi', 'equipment': 'P-101'},
    'K05': {'text': 'Prosedur LOTO belum dilakukan sebelum masuk tangki pembersihan.', 'category': 'Safety', 'priority': 'Kritis', 'equipment': None},
    'E03': {'text': 'Motor conveyor C-301 tidak start, trip di panel MCC.', 'category': 'Electrical', 'priority': 'Tinggi', 'equipment': 'C-301'},
    'M02': {'text': 'Kompresor K-202 overheating, suhu 95°C melebihi batas 85°C.', 'category': 'Mechanical', 'priority': 'Kritis', 'equipment': 'K-202'},
    'S01': {'text': 'Error ME21N: vendor master belum disetujui, PO tidak bisa dibuat.', 'category': 'SAP', 'priority': 'Sedang', 'equipment': None},
}

SOP_DB = {
    'SOP-MECH-001': 'LOTO Procedure: (1) Notify control room and obtain work permit. (2) Identify all energy sources. (3) Close isolation valves. (4) Apply LOTO lock and tag at each isolation point. (5) Verify zero energy state.',
    'SOP-MECH-005': 'Pump P-101 Maintenance: (1) Check vibration < 4.5 mm/s. (2) Inspect mechanical seal. (3) Check bearing temperature < 80°C. (4) Lubricate every 500 hours. (5) Replace bearing if vibration > 7 mm/s.',
    'SOP-ELEC-008': 'Motor Trip Investigation: (1) Check MCC panel for fault code. (2) Inspect motor winding resistance. (3) Check overload relay setting. (4) Inspect motor coupling alignment.',
    'SOP-MECH-012': 'Compressor K-202 Overtemp: (1) Reduce load to 60%. (2) Check cooling water flow > 15 L/min. (3) Inspect heat exchanger fouling. (4) Shutdown if temp > 100°C.',
    'SAP-GUIDE-001': 'SAP ME21N Purchase Order: (1) Verify vendor master (XK03). (2) Check vendor approval status. (3) Get procurement approval via workflow. (4) Re-run ME21N after approval.',
}

# --- Define tools using @tool decorator ---

@tool
def ticket_lookup(ticket_id: str) -> str:
    """Look up a maintenance or support ticket by ID (e.g. M01, K05, E03).
    Returns ticket text, category, priority, and linked equipment."""
    ticket = TICKET_DB.get(ticket_id.upper())
    if ticket:
        eq = ticket.get('equipment') or 'N/A'
        return (f"Ticket {ticket_id.upper()}: {ticket['text']} | "
                f"Kategori: {ticket['category']} | Prioritas: {ticket['priority']} | "
                f"Equipment: {eq}")
    return f"Ticket {ticket_id} tidak ditemukan dalam database."

@tool
def equipment_status(equipment_id: str) -> str:
    """Get real-time status of industrial equipment (e.g. P-101, K-202, C-301).
    Returns temperature, pressure, vibration, and next scheduled maintenance."""
    eq = EQUIPMENT_DB.get(equipment_id.upper())
    if eq:
        return (f"{eq['name']}: status={eq['status']}, "
                f"suhu={eq['temp_C']}°C, tekanan={eq['pressure_bar']} bar, "
                f"vibrasi={eq['vibration_mms']} mm/s, PM berikutnya: {eq['next_pm']}")
    return f"Equipment {equipment_id} tidak ditemukan."

@tool
def sop_search(query: str) -> str:
    """Search the industrial knowledge base for relevant SOPs and procedures.
    Uses keyword matching against SOP database. Returns matching SOP content.
    Examples: 'LOTO pump', 'compressor overtemp', 'SAP purchase order'."""
    query_lower = query.lower()
    matches = []
    for sop_id, content in SOP_DB.items():
        if any(kw in query_lower for kw in ['loto', 'lockout']) and 'MECH-001' in sop_id:
            matches.append((sop_id, content))
        elif any(kw in query_lower for kw in ['pump', 'p-101', 'pompa']) and 'MECH-005' in sop_id:
            matches.append((sop_id, content))
        elif any(kw in query_lower for kw in ['motor', 'trip', 'conveyor', 'elec']) and 'ELEC-008' in sop_id:
            matches.append((sop_id, content))
        elif any(kw in query_lower for kw in ['compress', 'k-202', 'overtemp', 'overheat']) and 'MECH-012' in sop_id:
            matches.append((sop_id, content))
        elif any(kw in query_lower for kw in ['sap', 'me21n', 'po', 'purchase', 'vendor']) and 'SAP-GUIDE' in sop_id:
            matches.append((sop_id, content))
    if matches:
        return '\n\n'.join([f'[Source: {sid}]\n{content}' for sid, content in matches[:2]])
    return f'Tidak ada SOP yang cocok untuk query: "{query}". Coba keyword yang berbeda.'

@tool
def create_work_order(equipment_id: str, description: str) -> str:
    """Create a maintenance Work Order (WO) in the CMMS/SAP PM system.
    Returns the generated WO number. Use only when maintenance action is confirmed."""
    import random
    wo_num = f'WO-2025-{random.randint(1000, 9999)}'
    return (f"Work Order dibuat: {wo_num} | Equipment: {equipment_id} | "
            f"Deskripsi: {description} | Status: Open | Assigned: Maintenance Team")

# List of tools for agent
TOOLS = [ticket_lookup, equipment_status, sop_search, create_work_order]

print('Tools defined:')
for t in TOOLS:
    name = t.name if hasattr(t, 'name') else t.__name__
    desc = (t.description if hasattr(t, 'description') else t.__doc__ or '')[:60]
    print(f'  - {name}: {desc}...')

In [ ]:
# CELL 2-B: Test tools directly (before wiring into agent)

print('=== Tool test: ticket_lookup ===')
result = ticket_lookup.invoke({'ticket_id': 'M01'}) if hasattr(ticket_lookup, 'invoke') else ticket_lookup('M01')
print(result)

print('\n=== Tool test: equipment_status ===')
result = equipment_status.invoke({'equipment_id': 'K-202'}) if hasattr(equipment_status, 'invoke') else equipment_status('K-202')
print(result)

print('\n=== Tool test: sop_search ===')
result = sop_search.invoke({'query': 'LOTO pump procedure'}) if hasattr(sop_search, 'invoke') else sop_search('LOTO pump procedure')
print(result[:400])

print('\n=== Tool test: sop_search (SAP) ===')
result = sop_search.invoke({'query': 'SAP ME21N vendor master'}) if hasattr(sop_search, 'invoke') else sop_search('SAP ME21N vendor master')
print(result[:400])

---
## Section 3 — ReAct Agents

### The ReAct Pattern (Reason + Act)

A **ReAct agent** interleaves reasoning and tool calls in a loop:

```
Thought: I need to check ticket M01 details first
Action: ticket_lookup({'ticket_id': 'M01'})
Observation: Ticket M01: Pompa P-101 bunyi berisik... Equipment: P-101
Thought: Equipment is P-101, let me check its current status
Action: equipment_status({'equipment_id': 'P-101'})
Observation: P-101: status=operational, vibrasi=3.1 mm/s...
Thought: I have enough information to draft a response
Final Answer: [grounded response with tool results]
```

The agent **decides** which tools to use and in what order — unlike a fixed chain.  
This is the key difference between a **chain** (fixed DAG) and an **agent** (dynamic).

In [ ]:
# CELL 3-A: Build ReAct agent with tool list

AGENT_SYSTEM = """Kamu adalah AI support agent untuk fasilitas manufaktur industri.
Kamu memiliki akses ke tools berikut untuk menangani tiket support:
- ticket_lookup: Cari detail tiket berdasarkan ID
- equipment_status: Cek status real-time equipment
- sop_search: Cari SOP dan prosedur dari knowledge base
- create_work_order: Buat Work Order di CMMS

Selalu:
1. Lookup tiket terlebih dahulu jika ada ticket ID
2. Cek status equipment jika tiket terkait equipment
3. Cari SOP yang relevan sebelum memberikan rekomendasi
4. Buat WO hanya jika maintenance action sudah jelas diperlukan
Berikan respons dalam Bahasa Indonesia yang profesional."""

if LANGCHAIN_AVAILABLE:
    try:
        from langchain.agents import create_react_agent, AgentExecutor
        from langchain_core.prompts import PromptTemplate
        
        # ReAct prompt template
        react_template = """Answer the following questions as best you can using these tools:
{tools}

Use this format:
Question: the input question
Thought: reason about what to do
Action: tool name
Action Input: tool input
Observation: tool result
... (repeat as needed)
Final Answer: final answer in Indonesian

System context: {system}
Tool names: {tool_names}
Question: {input}
{agent_scratchpad}"""
        
        react_prompt = PromptTemplate.from_template(react_template)
        react_agent = create_react_agent(llm, TOOLS, react_prompt)
        agent_executor = AgentExecutor(
            agent=react_agent, tools=TOOLS,
            verbose=True, max_iterations=5,
            handle_parsing_errors=True
        )
        print('✓ ReAct agent built with AgentExecutor')
        AGENT_BUILT = True
    except Exception as e:
        print(f'⚠  ReAct agent build failed: {e}')
        print('   Falling back to mock agent.')
        AGENT_BUILT = False
else:
    AGENT_BUILT = False
    print('⚠  Using mock agent (LangChain not available)')

# --- Mock agent for offline use ---
class MockReActAgent:
    """Simulates ReAct reasoning loop with deterministic tool calls."""
    def invoke(self, inputs):
        query = inputs.get('input', '')
        print(f'[Agent] Query: {query}')
        
        # Simulate ReAct loop
        ticket_ids = re.findall(r'\b([MKES]\d{2})\b', query.upper())
        equip_ids = re.findall(r'\b(P-101|K-202|C-301|HE-401)\b', query.upper())
        
        observations = []
        
        for tid in ticket_ids:
            obs = ticket_lookup(tid) if not hasattr(ticket_lookup, 'invoke') else ticket_lookup.invoke({'ticket_id': tid})
            print(f'[Agent] Thought: Look up ticket {tid}')
            print(f'[Agent] Action: ticket_lookup({tid!r})')
            print(f'[Agent] Observation: {obs[:100]}...')
            observations.append(obs)
        
        for eid in equip_ids:
            obs = equipment_status(eid) if not hasattr(equipment_status, 'invoke') else equipment_status.invoke({'equipment_id': eid})
            print(f'[Agent] Thought: Check equipment {eid}')
            print(f'[Agent] Action: equipment_status({eid!r})')
            print(f'[Agent] Observation: {obs[:100]}...')
            observations.append(obs)
        
        # SOP lookup
        sop_obs = sop_search(query) if not hasattr(sop_search, 'invoke') else sop_search.invoke({'query': query})
        print(f'[Agent] Thought: Search relevant SOPs')
        print(f'[Agent] Action: sop_search({query[:40]!r}...)')
        print(f'[Agent] Observation: {sop_obs[:100]}...')
        observations.append(sop_obs)
        
        final = llm.invoke([query + '\n\nContext: ' + ' | '.join(observations)]).content
        return {'output': final}

if not AGENT_BUILT:
    agent_executor = MockReActAgent()

print('Agent ready.')

In [ ]:
# CELL 3-B: Run the agent on a maintenance ticket

query = "Tolong bantu analisis tiket M01 tentang pompa P-101 yang berisik. Cek status equipment dan cari SOP yang relevan."

print('=== ReAct Agent Running ===' )
print(f'Query: {query}\n')

response = agent_executor.invoke({
    'input': query,
    'system': AGENT_SYSTEM
})

print(f"\n=== Final Answer ===")
print(response.get('output', response))

---
## Section 4 — Memory & Stateful Interactions

### Memory Types in LangChain

| Memory type | What it stores | Best for |
|---|---|---|
| **ConversationBufferMemory** | Full conversation history | Short sessions (<10 turns) |
| **ConversationSummaryMemory** | LLM-generated summary | Long sessions |
| **ConversationBufferWindowMemory** | Last k turns | Medium sessions |
| **VectorStoreRetrieverMemory** | Semantically relevant past messages | Large context |

**LangGraph approach (preferred):** Store conversation state in the graph `State` TypedDict.  
The `messages` field accumulates the full conversation — more explicit, easier to inspect.

In [ ]:
# CELL 4-A: Multi-turn support conversation with chat history

def run_support_conversation():
    """Simulate a multi-turn support agent conversation."""
    
    SUPPORT_SYSTEM = """Kamu adalah support agent teknis untuk fasilitas pabrik.
Ingat konteks percakapan sebelumnya dan berikan respons yang konsisten.
Jika teknisi bertanya lanjutan, gunakan informasi dari pertanyaan sebelumnya."""
    
    # Chat history accumulator
    history = []
    
    # Multi-turn conversation script
    turns = [
        "Halo, pompa P-101 di unit 3 bunyi aneh dan getaran tinggi sejak tadi pagi.",
        "Sudah berapa lama masalah ini terjadi? Dan ada perubahan operasi sebelumnya?",
        "Terjadi sejak maintenance terakhir 2 minggu lalu. Apakah bearing-nya perlu diganti?",
        "Kalau perlu ganti bearing, prosedur LOTO-nya bagaimana untuk pompa ini?",
    ]
    
    # Canned responses for multi-turn (used by mock or real LLM)
    canned = [
        "Terima kasih atas laporannya. Getaran tinggi pada P-101 bisa menandakan bearing aus atau masalah alignment. Nilai getaran saat ini berapa mm/s? Batas normal adalah < 4.5 mm/s (SOP-MECH-005).",
        "Baik, Anda menyebut pompa P-101 dengan getaran tinggi sejak maintenance 2 minggu lalu. Ini pola yang umum setelah maintenance jika alignment tidak sempurna. Cek alignment kopling terlebih dahulu sebelum memutuskan ganti bearing.",
        "Berdasarkan konteks sebelumnya (P-101, getaran tinggi pasca-maintenance), penggantian bearing mungkin diperlukan jika vibrasi > 7 mm/s. Lakukan pengukuran terlebih dahulu.",
        "Prosedur LOTO untuk P-101 (SOP-MECH-001): (1) Notifikasi control room, (2) Isolasi valve suction & discharge, (3) De-energize motor di MCC, (4) Pasang lock & tag personal, (5) Verifikasi zero energy. Jangan mulai pekerjaan sebelum semua langkah selesai.",
    ]
    
    print('=== Multi-Turn Support Conversation ===')
    print('Context: Maintenance technician reporting P-101 pump issue\n')
    
    for i, (user_msg, canned_resp) in enumerate(zip(turns, canned)):
        print(f'Turn {i+1}')
        print(f'Teknisi : {user_msg}')
        
        # Build messages with history
        messages = [{'role': 'system', 'content': SUPPORT_SYSTEM}]
        for h_user, h_assistant in history:
            messages.append({'role': 'user', 'content': h_user})
            messages.append({'role': 'assistant', 'content': h_assistant})
        messages.append({'role': 'user', 'content': user_msg})
        
        # LLM call (use canned for mock)
        if PROVIDER == 'mock':
            response_text = canned_resp
        else:
            if LANGCHAIN_AVAILABLE:
                msgs = [SystemMessage(content=SUPPORT_SYSTEM)]
                for h_user, h_assistant in history:
                    msgs.extend([HumanMessage(content=h_user), AIMessage(content=h_assistant)])
                msgs.append(HumanMessage(content=user_msg))
                response_text = llm.invoke(msgs).content
            else:
                response_text = llm.invoke([user_msg]).content
        
        print(f'Agent   : {response_text[:250]}')
        print()
        
        # Accumulate history
        history.append((user_msg, response_text))
    
    print(f'--- Conversation complete: {len(history)} turns, history tokens ≈ {sum(len(u)+len(a) for u,a in history)//4} ---')
    return history

conversation_history = run_support_conversation()

In [ ]:
# CELL 4-B: Memory strategies comparison

def summarize_conversation(history):
    """Simulate ConversationSummaryMemory — compress history to key facts."""
    if not history:
        return 'No conversation history.'
    
    # Collect all content
    all_text = ' '.join([u + ' ' + a for u, a in history])
    word_count = len(all_text.split())
    
    # Mock summary
    summary = (
        "[SUMMARY] Teknisi melaporkan masalah getaran tinggi pada pompa P-101 "
        "sejak maintenance 2 minggu lalu. Agent merekomendasikan cek alignment "
        "kopling dan pengukuran vibrasi. Jika vibrasi > 7 mm/s, ganti bearing. "
        "Prosedur LOTO SOP-MECH-001 harus diikuti sebelum pekerjaan."
    )
    return summary, word_count

summary, word_count = summarize_conversation(conversation_history)

print('=== Memory Strategy Comparison ===')
print()
print('Strategy 1: ConversationBufferMemory (full history)')
print(f'  Tokens (approx): {word_count} words ≈ {word_count * 1.3:.0f} tokens')
print(f'  Pros: Complete context')
print(f'  Cons: Token count grows unboundedly')
print()
print('Strategy 2: ConversationBufferWindowMemory (last k=2 turns)')
last_2 = conversation_history[-2:]
last_2_words = sum(len(u.split()) + len(a.split()) for u, a in last_2)
print(f'  Tokens (approx): {last_2_words} words ≈ {last_2_words * 1.3:.0f} tokens')
print(f'  Pros: Fixed token budget')
print(f'  Cons: Loses early context')
print()
print('Strategy 3: ConversationSummaryMemory (LLM-compressed)')
summary_words = len(summary.split())
print(f'  Tokens (approx): {summary_words} words ≈ {summary_words * 1.3:.0f} tokens')
print(f'  Pros: Compact, preserves key facts')
print(f'  Cons: Requires extra LLM call to summarize')
print()
print('Summary content:')
print(f'  {summary}')

---
## Section 5 — Introduction to LangGraph

### Why LangGraph?

LangChain chains are linear: A → B → C. Real workflows need:
- **Conditional routing**: if priority == 'Kritis', take escalation path
- **Loops**: retry if SOP not found; re-retrieve with refined query
- **Parallelism**: run equipment check and SOP search simultaneously
- **Human-in-the-loop**: pause and wait for human approval
- **Persistence**: resume interrupted workflow from checkpoint

**LangGraph** models this as a **directed graph** where:
- **Nodes** = processing functions (classify, retrieve, draft, escalate)
- **Edges** = transitions between nodes (including conditional edges)
- **State** = a TypedDict passed through the graph, accumulating results

```
START → classify_ticket → [Kritis?] ──yes──→ escalate → END
                                   └──no───→ retrieve_sop → draft_response → END
```

In [ ]:
# CELL 5-A: Simple linear LangGraph — classify ticket

from typing import TypedDict, List, Optional

# --- State definition ---
class TicketState(TypedDict):
    ticket_id: str
    ticket_text: str
    category: Optional[str]
    priority: Optional[str]
    equipment_id: Optional[str]
    equipment_status: Optional[str]
    sop_content: Optional[str]
    draft_response: Optional[str]
    escalated: bool
    messages: List[str]

# --- Node functions ---
def classify_node(state: TicketState) -> dict:
    """Node 1: Classify the ticket category and priority."""
    text = state['ticket_text'].lower()
    
    # Rule-based classification (mock — real version uses LLM chain)
    if any(kw in text for kw in ['loto', 'keselamatan', 'safety', 'apd', 'k3']):
        category, priority = 'Safety', 'Kritis'
    elif any(kw in text for kw in ['pompa', 'kompresor', 'bearing', 'getaran', 'overtemp']):
        category, priority = 'Mechanical', 'Tinggi'
    elif any(kw in text for kw in ['motor', 'electrical', 'listrik', 'trip', 'mcc']):
        category, priority = 'Electrical', 'Tinggi'
    elif any(kw in text for kw in ['sap', 'me21n', 'po', 'vendor', 'erp']):
        category, priority = 'SAP', 'Sedang'
    elif any(kw in text for kw in ['vpn', 'network', 'server', 'it', 'historian']):
        category, priority = 'IT', 'Sedang'
    else:
        category, priority = 'General', 'Rendah'
    
    # Extract equipment ID
    eq_match = re.search(r'\b(P-101|K-202|C-301|HE-401)\b', state['ticket_text'].upper())
    equipment_id = eq_match.group(1) if eq_match else None
    
    log = f'classify: {category}/{priority}' + (f'/equipment={equipment_id}' if equipment_id else '')
    
    return {
        'category': category,
        'priority': priority,
        'equipment_id': equipment_id,
        'messages': state.get('messages', []) + [log]
    }

def fetch_equipment_node(state: TicketState) -> dict:
    """Node 2: Fetch equipment status if equipment ID is available."""
    eq_id = state.get('equipment_id')
    if not eq_id:
        return {'equipment_status': None, 'messages': state['messages'] + ['fetch_equipment: skipped (no equipment)']}
    
    eq_data = EQUIPMENT_DB.get(eq_id, {})
    status_str = f"{eq_id}: {eq_data.get('status','unknown')}, temp={eq_data.get('temp_C','?')}°C, vib={eq_data.get('vibration_mms','?')} mm/s"
    return {
        'equipment_status': status_str,
        'messages': state['messages'] + [f'fetch_equipment: {status_str[:60]}']
    }

def retrieve_sop_node(state: TicketState) -> dict:
    """Node 3: Retrieve relevant SOP from knowledge base."""
    query = f"{state.get('category','')} {state['ticket_text'][:100]}"
    sop_result = sop_search(query) if not hasattr(sop_search, 'invoke') else sop_search.invoke({'query': query})
    return {
        'sop_content': sop_result,
        'messages': state['messages'] + [f'retrieve_sop: found={len(sop_result)} chars']
    }

def draft_response_node(state: TicketState) -> dict:
    """Node 4: Draft the support response using all gathered context."""
    context = f"""
Tiket: {state['ticket_id']} — {state['ticket_text']}
Kategori: {state.get('category')} | Prioritas: {state.get('priority')}
Status Equipment: {state.get('equipment_status', 'N/A')}
SOP Relevan: {(state.get('sop_content') or '')[:300]}
"""
    draft = llm.invoke([context + '\nBuat balasan support profesional dalam Bahasa Indonesia.']).content
    return {
        'draft_response': draft,
        'messages': state['messages'] + ['draft_response: complete']
    }

# --- Build simple linear graph ---
if LANGGRAPH_AVAILABLE:
    simple_graph = StateGraph(TicketState)
    simple_graph.add_node('classify', classify_node)
    simple_graph.add_node('fetch_equipment', fetch_equipment_node)
    simple_graph.add_node('retrieve_sop', retrieve_sop_node)
    simple_graph.add_node('draft_response', draft_response_node)
    
    simple_graph.set_entry_point('classify')
    simple_graph.add_edge('classify', 'fetch_equipment')
    simple_graph.add_edge('fetch_equipment', 'retrieve_sop')
    simple_graph.add_edge('retrieve_sop', 'draft_response')
    simple_graph.add_edge('draft_response', END)
    
    compiled_graph = simple_graph.compile()
    print('✓ LangGraph linear graph compiled')
else:
    # Mock graph executor
    class MockGraph:
        def invoke(self, state):
            state = {**state, 'escalated': state.get('escalated', False), 'messages': []}
            state.update(classify_node(state))
            state.update(fetch_equipment_node(state))
            state.update(retrieve_sop_node(state))
            state.update(draft_response_node(state))
            return state
    compiled_graph = MockGraph()
    print('ℹ  Using mock graph (LangGraph not installed)')

# --- Run the graph ---
initial_state = {
    'ticket_id': 'M01',
    'ticket_text': 'Pompa sentrifugal P-101 mengeluarkan bunyi berisik dan getaran tinggi. Bearing mungkin aus.',
    'category': None, 'priority': None, 'equipment_id': None,
    'equipment_status': None, 'sop_content': None, 'draft_response': None,
    'escalated': False, 'messages': []
}

print('\nRunning linear graph on ticket M01...')
result = compiled_graph.invoke(initial_state)

print(f"\n--- Graph execution trace ---")
for msg in result.get('messages', []):
    print(f'  [{msg}]')
print(f"\nCategory  : {result.get('category')} | Priority: {result.get('priority')}")
print(f"Equipment : {result.get('equipment_status', 'N/A')[:80]}")
print(f"Draft     : {result.get('draft_response', '')[:200]}")

---
## Section 6 — Multi-Step & Conditional Workflow

### Conditional Edges in LangGraph

A **conditional edge** routes the graph to different nodes based on the current state:

```python
graph.add_conditional_edges(
    'classify',
    route_by_priority,          # function returns node name
    {'escalate': 'escalate_node', 'continue': 'retrieve_sop'}
)
```

This is the key difference from a linear chain — the graph can branch dynamically  
based on the ticket's category, priority, or any field in the state.

In [ ]:
# CELL 6-A: Conditional routing — triage graph with escalation path

def escalate_node(state: TicketState) -> dict:
    """Escalation node: notify supervisor for critical tickets."""
    msg = (f"[ESKALASI] Tiket {state['ticket_id']} | Kategori: {state.get('category')} | "
           f"Prioritas: {state.get('priority')}. "
           f"Supervisor notified: supervisor@plant.id | ETA: 30 menit.")
    return {
        'escalated': True,
        'draft_response': msg,
        'messages': state['messages'] + ['escalate: supervisor notified']
    }

# --- Routing function (returns node name to go to next) ---
def route_after_classify(state: TicketState) -> str:
    """Route: Kritis tickets → escalate immediately. Others → continue processing."""
    if state.get('priority') == 'Kritis':
        return 'escalate'
    return 'fetch_equipment'

# --- Build conditional graph ---
if LANGGRAPH_AVAILABLE:
    triage_graph = StateGraph(TicketState)
    
    # Add all nodes
    triage_graph.add_node('classify', classify_node)
    triage_graph.add_node('fetch_equipment', fetch_equipment_node)
    triage_graph.add_node('retrieve_sop', retrieve_sop_node)
    triage_graph.add_node('draft_response', draft_response_node)
    triage_graph.add_node('escalate', escalate_node)
    
    # Set entry point
    triage_graph.set_entry_point('classify')
    
    # Conditional edge after classify
    triage_graph.add_conditional_edges(
        'classify',
        route_after_classify,
        {'escalate': 'escalate', 'fetch_equipment': 'fetch_equipment'}
    )
    
    # Normal path edges
    triage_graph.add_edge('fetch_equipment', 'retrieve_sop')
    triage_graph.add_edge('retrieve_sop', 'draft_response')
    triage_graph.add_edge('draft_response', END)
    triage_graph.add_edge('escalate', END)
    
    triage_compiled = triage_graph.compile()
    print('✓ Triage graph with conditional routing compiled')
else:
    class MockTriageGraph:
        def invoke(self, state):
            state = {**state, 'escalated': state.get('escalated', False), 'messages': []}
            state.update(classify_node(state))
            route = route_after_classify(state)
            if route == 'escalate':
                state.update(escalate_node(state))
            else:
                state.update(fetch_equipment_node(state))
                state.update(retrieve_sop_node(state))
                state.update(draft_response_node(state))
            return state
    triage_compiled = MockTriageGraph()
    print('ℹ  Using mock triage graph')

# --- Test on multiple tickets to see routing in action ---
test_tickets = [
    {'ticket_id': 'K05', 'ticket_text': 'Prosedur LOTO belum dilakukan sebelum teknisi masuk ke tangki pembersihan. Bahaya keselamatan.'},
    {'ticket_id': 'M01', 'ticket_text': 'Pompa sentrifugal P-101 bunyi berisik dan getaran tinggi. Bearing mungkin aus.'},
    {'ticket_id': 'S01', 'ticket_text': 'Error ME21N SAP: vendor master belum disetujui, PO tidak bisa dibuat.'},
]

print()
for t in test_tickets:
    state = {**t, 'category': None, 'priority': None, 'equipment_id': None,
             'equipment_status': None, 'sop_content': None, 'draft_response': None,
             'escalated': False, 'messages': []}
    result = triage_compiled.invoke(state)
    path = 'ESCALATED' if result.get('escalated') else 'NORMAL'
    print(f"Ticket {result['ticket_id']:4s} → {result.get('category'):12s} / {result.get('priority'):6s} → {path}")
    print(f"  Trace: {' → '.join(result.get('messages', [])[:3])}")
    print()

In [ ]:
# CELL 6-B: Run triage graph on all 5 sample tickets — compare routing

ALL_TEST_TICKETS = [
    {'ticket_id': 'M01', 'ticket_text': 'Pompa sentrifugal P-101 bunyi berisik dan getaran tinggi. Bearing mungkin aus.'},
    {'ticket_id': 'M02', 'ticket_text': 'Kompresor K-202 overheating suhu 95°C melebihi batas 85°C. Perlu shutdown segera.'},
    {'ticket_id': 'K05', 'ticket_text': 'Prosedur LOTO belum dilakukan. Teknisi masuk tangki tanpa APD lengkap. Safety K3!'},
    {'ticket_id': 'S01', 'ticket_text': 'Error ME21N SAP: vendor master belum disetujui procurement. PO tidak bisa dibuat.'},
    {'ticket_id': 'N04', 'ticket_text': 'Koneksi VPN ke plant network terputus setiap 2 jam. Teknisi remote tidak bisa akses DCS.'},
]

print(f'{"ID":<6} {"Category":<14} {"Priority":<8} {"Route":<12} {"Escalated"}')
print('-' * 60)

routing_results = []
for t in ALL_TEST_TICKETS:
    state = {**t, 'category': None, 'priority': None, 'equipment_id': None,
             'equipment_status': None, 'sop_content': None, 'draft_response': None,
             'escalated': False, 'messages': []}
    result = triage_compiled.invoke(state)
    route = 'ESCALATE' if result.get('escalated') else 'NORMAL'
    routing_results.append(result)
    print(f"{result['ticket_id']:<6} {result.get('category','?'):<14} {result.get('priority','?'):<8} {route:<12} {result.get('escalated')}")

escalated_count = sum(1 for r in routing_results if r.get('escalated'))
print(f'\nSummary: {escalated_count}/{len(ALL_TEST_TICKETS)} tickets escalated')

---
## Section 7 — Error Handling & Human-in-the-Loop

### Production-Grade Graphs Need:

1. **Retry logic** — if SOP not found, try with a different query
2. **Fallbacks** — if LLM fails, use rule-based response
3. **Human-in-the-Loop (HITL)** — pause graph and wait for human approval
4. **Timeout handling** — max retries before giving up

**LangGraph `interrupt_before`:** Pause the graph before a node, return control  
to the application, receive human input, then resume the graph from the checkpoint.

In [ ]:
# CELL 7-A: Retry logic — re-query SOP if not found

def retrieve_sop_with_retry(state: TicketState, max_retries: int = 2) -> dict:
    """SOP retrieval with retry and query refinement."""
    
    queries = [
        f"{state.get('category', '')} {state['ticket_text'][:80]}",           # Query 1: category + text
        f"{state.get('equipment_id', '')} maintenance procedure",              # Query 2: equipment-specific
        f"{state.get('category', '').lower()} safety procedure indonesia",    # Query 3: broader
    ]
    
    for attempt, query in enumerate(queries[:max_retries + 1], start=1):
        result = sop_search(query) if not hasattr(sop_search, 'invoke') else sop_search.invoke({'query': query})
        if 'Tidak ada SOP' not in result and len(result) > 50:
            print(f'  SOP found on attempt {attempt} with query: "{query[:50]}..."')
            return {
                'sop_content': result,
                'messages': state['messages'] + [f'retrieve_sop: found on attempt {attempt}']
            }
        print(f'  Attempt {attempt}: no SOP found, refining query...')
    
    # Fallback: no SOP found after retries
    fallback = 'Tidak ada SOP spesifik ditemukan. Rujuk ke supervisor dan manual equipment.'
    return {
        'sop_content': fallback,
        'messages': state['messages'] + ['retrieve_sop: fallback (no SOP found after retries)']
    }

# Test retry logic
print('Testing retry logic...')
test_state = {
    'ticket_id': 'M02', 'ticket_text': 'Kompresor K-202 overheating suhu 95°C.',
    'category': 'Mechanical', 'priority': 'Kritis', 'equipment_id': 'K-202',
    'equipment_status': None, 'sop_content': None, 'draft_response': None,
    'escalated': False, 'messages': []
}
result = retrieve_sop_with_retry(test_state)
print(f"SOP content ({len(result['sop_content'])} chars):")
print(result['sop_content'][:300])

In [ ]:
# CELL 7-B: Human-in-the-Loop — approval node simulation

def human_approval_node(state: TicketState) -> dict:
    """HITL node: pause and request human approval for WO creation.
    In production: sends notification and waits for supervisor response via API.
    In notebook: simulates approval with configurable response."""
    
    # Construct approval request
    request = (
        f"[APPROVAL REQUIRED]\n"
        f"Ticket: {state['ticket_id']} | Category: {state.get('category')} | Priority: {state.get('priority')}\n"
        f"Proposed action: Create Work Order for {state.get('equipment_id', 'N/A')}\n"
        f"SOP reference: {(state.get('sop_content') or '')[:100]}...\n"
        f"Awaiting supervisor approval."
    )
    
    print(request)
    
    # Simulate: in real system this would pause (interrupt_before) and wait
    # For notebook: auto-approve non-critical, manual prompt for critical
    is_critical = state.get('priority') == 'Kritis'
    
    if is_critical:
        # Simulate supervisor approval (in real system: HTTP webhook or message queue)
        simulated_approval = True  # Change to False to test rejection path
        print(f'\n[SIMULATED] Supervisor response: {"APPROVED" if simulated_approval else "REJECTED"}')
    else:
        simulated_approval = True  # Non-critical: auto-approve
        print(f'\n[AUTO-APPROVED] Non-critical ticket — work order auto-approved')
    
    return {
        'messages': state['messages'] + [
            f'human_approval: {"approved" if simulated_approval else "rejected"}'
        ],
        # Store approval decision in messages for downstream nodes
        'draft_response': state.get('draft_response', '') + f'\n[Approval: {"APPROVED" if simulated_approval else "REJECTED"}]'
    }

def create_wo_node(state: TicketState) -> dict:
    """Create Work Order after human approval."""
    eq_id = state.get('equipment_id') or 'GENERAL'
    desc = f"{state.get('category')} issue: {state['ticket_text'][:80]}"
    
    wo_result = create_work_order(eq_id, desc) if not hasattr(create_work_order, 'invoke') else \
                create_work_order.invoke({'equipment_id': eq_id, 'description': desc})
    
    return {
        'draft_response': (state.get('draft_response') or '') + f'\n{wo_result}',
        'messages': state['messages'] + [f'create_wo: {wo_result[:50]}']
    }

# Simulate HITL flow
print('=== Human-in-the-Loop Simulation ===')
print()
hitl_state = {
    'ticket_id': 'M02', 'ticket_text': 'Kompresor K-202 overheating 95°C. Shutdown diperlukan.',
    'category': 'Mechanical', 'priority': 'Kritis', 'equipment_id': 'K-202',
    'equipment_status': 'K-202: warning, 95°C', 'sop_content': 'SOP-MECH-012: Reduce load to 60%...',
    'draft_response': 'Initial draft: Tiket M02 memerlukan perhatian segera.',
    'escalated': False, 'messages': ['classify: Mechanical/Kritis', 'fetch_equipment: done', 'retrieve_sop: found']
}

# Run approval node
approval_result = human_approval_node(hitl_state)
print()

# Run WO creation if approved
merged = {**hitl_state, **approval_result}
wo_result = create_wo_node(merged)
print('WO Creation result:')
print(wo_result['draft_response'][-200:])

In [ ]:
# CELL 7-C: Full graph with HITL interrupt (LangGraph interrupt_before)

if LANGGRAPH_AVAILABLE:
    from langgraph.checkpoint.memory import MemorySaver
    
    hitl_graph = StateGraph(TicketState)
    hitl_graph.add_node('classify', classify_node)
    hitl_graph.add_node('fetch_equipment', fetch_equipment_node)
    hitl_graph.add_node('retrieve_sop', lambda s: retrieve_sop_with_retry(s))
    hitl_graph.add_node('human_approval', human_approval_node)
    hitl_graph.add_node('create_wo', create_wo_node)
    hitl_graph.add_node('draft_response', draft_response_node)
    
    hitl_graph.set_entry_point('classify')
    hitl_graph.add_conditional_edges('classify', route_after_classify,
                                      {'escalate': 'draft_response', 'fetch_equipment': 'fetch_equipment'})
    hitl_graph.add_edge('fetch_equipment', 'retrieve_sop')
    hitl_graph.add_edge('retrieve_sop', 'human_approval')
    hitl_graph.add_edge('human_approval', 'create_wo')
    hitl_graph.add_edge('create_wo', 'draft_response')
    hitl_graph.add_edge('draft_response', END)
    
    # Compile with checkpointer for HITL resume capability
    memory = MemorySaver()
    hitl_compiled = hitl_graph.compile(
        checkpointer=memory,
        interrupt_before=['human_approval']  # Pause before approval node
    )
    print('✓ HITL graph compiled with interrupt_before=[human_approval]')
    print('  In production: graph.invoke() returns at human_approval node')
    print('  Human reviews, then: graph.invoke(None, config) to resume')
else:
    print('ℹ  LangGraph not installed — HITL interrupt requires langgraph package')
    print('   The human_approval_node() function above simulates the pattern.')
    print('   Install: pip install langgraph')

---
## Section 8 — Use Case: Autonomous Support Agent Pipeline

### Full Pipeline: classify → retrieve (RAG) → draft → escalate or close

Building on Day 3's RAG pipeline, we now wrap it in a LangGraph agent  
that handles the complete ticket lifecycle:

```
                    ┌─────────────────────────────────────┐
Ticket              │      AUTONOMOUS SUPPORT AGENT       │
  │                 │                                     │
  ▼                 │  classify ──→ fetch_equipment       │
  INPUT             │      │             │                │
                    │      ▼             ▼                │
                    │  [Kritis?]    retrieve_sop          │
                    │   yes│no          │                 │
                    │    │  └──────────→│                 │
                    │    ▼             ▼                  │
                    │  escalate   human_approval          │
                    │    │             │                  │
                    │    └────────────→│                  │
                    │                 ▼                   │
                    │           draft_response            │
                    │                 │                   │
                    └─────────────────┼───────────────────┘
                                      ▼
                               Ticket Closed
```

In [ ]:
# CELL 8-A: Autonomous support agent — full pipeline on all test tickets

def run_autonomous_pipeline(tickets: list) -> pd.DataFrame:
    """Run the full triage graph on a list of tickets and return results DataFrame."""
    results = []
    
    for t in tickets:
        state = {
            'ticket_id': t['ticket_id'],
            'ticket_text': t['ticket_text'],
            'category': None, 'priority': None, 'equipment_id': None,
            'equipment_status': None, 'sop_content': None, 'draft_response': None,
            'escalated': False, 'messages': []
        }
        
        start = time.time()
        result = triage_compiled.invoke(state)
        elapsed = time.time() - start
        
        results.append({
            'ticket_id': result['ticket_id'],
            'category': result.get('category', '?'),
            'priority': result.get('priority', '?'),
            'equipment_id': result.get('equipment_id') or 'N/A',
            'escalated': result.get('escalated', False),
            'sop_found': bool(result.get('sop_content') and 'Tidak ada' not in result.get('sop_content', '')),
            'response_len': len(result.get('draft_response') or ''),
            'latency_s': round(elapsed, 3),
            'steps': len(result.get('messages', []))
        })
    
    return pd.DataFrame(results)

# Full 40-ticket dataset (same tickets as Days 1, 2, 3)
FULL_TICKETS = [
    {'ticket_id': 'M01', 'ticket_text': 'Pompa sentrifugal P-101 mengeluarkan bunyi berisik dan getaran tinggi. Bearing mungkin aus.'},
    {'ticket_id': 'M02', 'ticket_text': 'Kompresor K-202 overheating. Suhu mencapai 95°C, melebihi batas 85°C.'},
    {'ticket_id': 'M03', 'ticket_text': 'Kebocoran oli pada gearbox unit mixing M-501. Perlu penggantian seal.'},
    {'ticket_id': 'M04', 'ticket_text': 'Valve control HV-201 tidak merespons sinyal dari DCS. Posisi stuck di 50%.'},
    {'ticket_id': 'M05', 'ticket_text': 'Coupling pompa P-203 aus, perlu penggantian segera sebelum breakdown total.'},
    {'ticket_id': 'E01', 'ticket_text': 'Panel MCC unit 2 mengalami trip berulang. Overload relay sering aktif.'},
    {'ticket_id': 'E02', 'ticket_text': 'Transformator T-101 mengeluarkan suara dengung abnormal. Perlu inspeksi.'},
    {'ticket_id': 'E03', 'ticket_text': 'Motor penggerak conveyor C-301 tidak mau start. Indikator trip di panel MCC.'},
    {'ticket_id': 'E04', 'ticket_text': 'Ground fault terdeteksi di circuit breaker CB-405. Earth leakage 45mA.'},
    {'ticket_id': 'E05', 'ticket_text': 'UPS sistem kontrol kehilangan kapasitas baterai. Runtime hanya 8 menit.'},
    {'ticket_id': 'K01', 'ticket_text': 'Baju pelindung panas tidak tersedia di area furnace. APD tidak lengkap.'},
    {'ticket_id': 'K02', 'ticket_text': 'Alat pemadam kebakaran di area storage kedaluwarsa bulan lalu.'},
    {'ticket_id': 'K03', 'ticket_text': 'Papan rambu bahaya di area tangki B-102 rusak dan tidak terbaca.'},
    {'ticket_id': 'K04', 'ticket_text': 'Prosedur hot work permit tidak diikuti saat pengelasan di area terbuka.'},
    {'ticket_id': 'K05', 'ticket_text': 'Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan.'},
    {'ticket_id': 'S01', 'ticket_text': 'Error ME21N: vendor master belum disetujui procurement. PO tidak bisa dibuat.'},
    {'ticket_id': 'S02', 'ticket_text': 'SAP PM Work Order WO-2024-1205 tidak bisa di-close karena konfirmasi operasi belum lengkap.'},
    {'ticket_id': 'S03', 'ticket_text': 'Transaksi MIGO untuk penerimaan barang gagal: material master tidak ditemukan.'},
    {'ticket_id': 'S04', 'ticket_text': 'Cost center di SAP CO tidak sesuai untuk alokasi biaya maintenance bulan ini.'},
    {'ticket_id': 'S05', 'ticket_text': 'User SAP tidak bisa akses modul PM setelah reset password. Authorization object hilang.'},
    {'ticket_id': 'N01', 'ticket_text': 'Server historian kehilangan data 3 jam karena disk penuh. Buffer overflow.'},
    {'ticket_id': 'N02', 'ticket_text': 'Firewall memblokir komunikasi antara OT network dan ERP SAP. Latency tinggi.'},
    {'ticket_id': 'N03', 'ticket_text': 'SCADA tidak bisa terhubung ke PLC unit 4 setelah pembaruan firmware.'},
    {'ticket_id': 'N04', 'ticket_text': 'Koneksi VPN ke plant network terputus setiap 2 jam. Remote access terganggu.'},
    {'ticket_id': 'N05', 'ticket_text': 'Backup database DCS gagal sejak seminggu lalu. Storage NAS tidak terbaca.'},
]

print('Running autonomous pipeline on 25 tickets...')
pipeline_df = run_autonomous_pipeline(FULL_TICKETS)

print(f'\nPipeline complete: {len(pipeline_df)} tickets processed')
print(pipeline_df.to_string(index=False))

In [ ]:
# CELL 8-B: Pipeline analysis — routing decisions and coverage

print('=== Pipeline Summary ===')
print(f'Total tickets processed : {len(pipeline_df)}')
print(f'Escalated (Kritis)      : {pipeline_df["escalated"].sum()}')
print(f'Normal path             : {(~pipeline_df["escalated"]).sum()}')
print(f'SOP found               : {pipeline_df["sop_found"].sum()}')
print(f'Avg latency             : {pipeline_df["latency_s"].mean():.3f}s')
print(f'Avg steps per ticket    : {pipeline_df["steps"].mean():.1f}')
print()

print('=== By Category ===')
cat_summary = pipeline_df.groupby('category').agg(
    count=('ticket_id', 'count'),
    escalated=('escalated', 'sum'),
    sop_found=('sop_found', 'sum'),
    avg_latency=('latency_s', 'mean')
).round(3)
print(cat_summary.to_string())

print()
print('=== By Priority ===')
pri_summary = pipeline_df.groupby('priority').agg(
    count=('ticket_id', 'count'),
    escalated=('escalated', 'sum')
)
print(pri_summary.to_string())

In [ ]:
# CELL 8-C: Sample full agent response — single ticket end-to-end

# Show a complete single-ticket run with all details
demo_ticket = {
    'ticket_id': 'M02',
    'ticket_text': 'Kompresor K-202 overheating. Suhu mencapai 95°C, melebihi batas operasi 85°C. Perlu tindakan segera.'
}

state = {
    **demo_ticket,
    'category': None, 'priority': None, 'equipment_id': None,
    'equipment_status': None, 'sop_content': None, 'draft_response': None,
    'escalated': False, 'messages': []
}

result = triage_compiled.invoke(state)

print('=' * 60)
print('AUTONOMOUS SUPPORT AGENT — FULL TICKET TRACE')
print('=' * 60)
print(f"Input : {demo_ticket['ticket_id']} — {demo_ticket['ticket_text']}")
print()
print('Graph execution steps:')
for i, step in enumerate(result.get('messages', []), 1):
    print(f'  Step {i}: {step}')
print()
print(f"Category  : {result.get('category')}")
print(f"Priority  : {result.get('priority')}")
print(f"Equipment : {result.get('equipment_id', 'N/A')}")
print(f"Escalated : {result.get('escalated')}")
print()
print('SOP Retrieved:')
print(f"  {(result.get('sop_content') or 'N/A')[:200]}")
print()
print('Draft Response:')
print(result.get('draft_response', 'N/A'))

---
## Section 9 — Next Steps & Day 5 Preview

### What We Built Tonight

| Step | What we built | Key concept |
|------|--------------|-------------|
| Section 1 | `ticket_chain` LCEL pipeline | Runnable protocol, pipe operator |
| Section 2 | 4 industrial tools | @tool decorator, JSON schema |
| Section 3 | ReAct agent with tools | Reason + Act loop, AgentExecutor |
| Section 4 | Multi-turn conversation | Chat history, memory strategies |
| Section 5 | Linear LangGraph | StateGraph, nodes, edges, TypedDict |
| Section 6 | Conditional routing graph | Conditional edges, triage path |
| Section 7 | Retry + HITL graph | Retry logic, interrupt_before |
| Section 8 | Autonomous support agent | Full pipeline on 25 tickets |

### Day 5 Preview — Deployment & Production Systems

| Tonight (Day 4) | Day 5 |
|---|---|
| Local notebook agent | Production API service (FastAPI) |
| Anthropic/OpenAI API | Local LLM with vLLM or Ollama |
| Mock CMMS tools | Real API integrations |
| Single-user notebook | Multi-user service with auth |
| No monitoring | LangSmith tracing, cost tracking |

### Homework Exercises

1. **Tool extension:** Add a `send_escalation_email(ticket_id, supervisor_email)` tool  
   and wire it into the escalation node
2. **Memory upgrade:** Replace the list-based history with `ConversationSummaryMemory`  
   and observe how the summary changes across turns
3. **Graph extension:** Add a `verify_sop_relevance` node after `retrieve_sop` that  
   checks if the retrieved SOP is actually relevant (cosine > 0.7 threshold)
4. **Evaluation:** Run the pipeline on all 40 tickets and measure: % with SOP found,  
   % escalated, average response length
5. **RAG integration:** Replace the keyword-based `sop_search` tool with the real  
   `rag_query()` function from Day 3 (requires FAISS index from Day 3 notebook)